# 🎯 Adaptive Kill-Chain Router (AKCR) — Google Colab GPU Runner
**Hybrid Reinforcement Learning & AlignTune DPO for Autonomous Red-Teaming (CybORG CAGE-4)**

This notebook executes the remediated 4-Quadrant Kill-Chain Benchmark on a Google Colab GPU (`T4`, `L4`, or `A100`).
You can either run this notebook directly in Google Colab or connect your local IDE to a Colab GPU runtime and execute `scripts/setup_colab.sh` and `scripts/run_colab_pipeline.py`.

In [2]:
# 1. Verify GPU & Clone/Navigate to Repository
!nvidia-smi
import os

if not os.path.exists('src/router/killchain_router.py'):
    if not os.path.exists('LexsiLabs'):
        !git clone https://github.com/sawantviraj465/LexsiLabs.git
    %cd LexsiLabs
print('Working directory:', os.getcwd())

/bin/bash: line 1: nvidia-smi: command not found
Cloning into 'LexsiLabs'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'LexsiLabs'
/content
Working directory: /content


In [ ]:
# 2. (Optional) Mount Google Drive for Persistent Checkpointing Across Disconnections
USE_GOOGLE_DRIVE = True  # Set True on Colab web UI to auto-backup checkpoints to Drive
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/LexsiLabs_Results'

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    print(f'✅ Google Drive mounted at: {DRIVE_BACKUP_DIR}')
else:
    DRIVE_BACKUP_DIR = ''
    print('ℹ️ Running with local Colab storage (results/).')

In [ ]:
# 3. Install Dependencies & CybORG CAGE-4 Environment
!bash scripts/setup_colab.sh

In [ ]:
# 4. Configure Hugging Face Token (Required for gated Llama-3.1-8B models)
# Either add HF_TOKEN to Google Colab Secrets (key icon on left sidebar) or paste below:
import os
try:
    from google.colab import userdata
    if userdata.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        print('✅ Loaded HF_TOKEN from Colab Secrets.')
except Exception:
    pass

if not os.environ.get('HF_TOKEN'):
    from getpass import getpass
    token_in = getpass('Enter your Hugging Face token (or press Enter for ungated models like Qwen): ').strip()
    if token_in:
        os.environ['HF_TOKEN'] = token_in

In [ ]:
# 5. Run Unit Tests to Verify Router Fixes Before GPU Sweep
!python3 -m pytest tests/test_router.py -v

In [ ]:
# 6. Launch End-to-End AlignTune DPO Training + 30-Episode 4-Quadrant Benchmark
MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'  # or 'Qwen/Qwen2.5-7B-Instruct'
EPISODES = 30
MAX_STEPS = 120

from scripts.run_colab_pipeline import run_colab_pipeline
run_colab_pipeline(
    model_name=MODEL_NAME,
    episodes=EPISODES,
    max_steps=MAX_STEPS,
    force_train=False,
    fresh=True,
    drive_backup_dir=DRIVE_BACKUP_DIR,
)

In [ ]:
# 7. Display Generated Publication Chart & Archive Results for Download
from IPython.display import Image, Markdown, display
import shutil

chart_path = 'results/meta_llama_3_1_8b_v2/killchain_ablation_chart.png'
md_path = 'results/meta_llama_3_1_8b_v2/ablation_results.md'
if os.path.exists(md_path):
    with open(md_path) as f:
        display(Markdown(f.read()))
if os.path.exists(chart_path):
    display(Image(filename=chart_path))

shutil.make_archive('lexsilabs_colab_results', 'zip', 'results')
print('📦 Archived all results and LoRA checkpoints to lexsilabs_colab_results.zip')